# 05 · Modelos
Regla: se ajusta en `train`, se afina en `valid` y `test` se usa una sola vez, al final. Primero un LightGBM como referencia de capacidad predictiva; después un scorecard WoE + regresión logística como modelo interpretable.

In [0]:
%pip install lightgbm -q

In [0]:
dbutils.library.restartPython()

In [0]:
# Cargar y separar. Las columnas de texto pasan a tipo category, que LightGBM maneja nativamente sin necesidad de one-hot. Los nulos también los maneja solo:
import pandas as pd, numpy as np
ESQ = "workspace.credit_risk"

pdf = spark.table(f"{ESQ}.gold_features").toPandas()

y_col, id_col, split_col = "TARGET", "SK_ID_CURR", "split"
cat_cols = [c for c in pdf.columns if pdf[c].dtype == "object" and c != split_col]
for c in cat_cols:
    pdf[c] = pdf[c].astype("category")
features = [c for c in pdf.columns if c not in (y_col, id_col, split_col)]

tr = pdf[pdf[split_col] == "train"]
va = pdf[pdf[split_col] == "valid"]
te = pdf[pdf[split_col] == "test"]     

X_tr, y_tr = tr[features], tr[y_col]
X_va, y_va = va[features], va[y_col]
print(f"{len(features)} variables ({len(cat_cols)} categóricas)")
print({"train": len(tr), "valid": len(va), "test": len(te)})

In [0]:
# Entrenar, evaluar y registrar en MLflow:
import lightgbm as lgb, mlflow
from sklearn.metrics import roc_auc_score

usuario = spark.sql("select current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/home_credit_pd")

params = dict(objective="binary", metric="auc", learning_rate=0.05, n_estimators=2000,
              num_leaves=31, min_child_samples=100, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.5, reg_lambda=5.0, random_state=42, verbose=-1)

with mlflow.start_run(run_name="lgbm_baseline"):
    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)])

    auc_tr = roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1])
    auc_va = roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])
    mlflow.log_params({**params, "best_iteration": m.best_iteration_})
    mlflow.log_metrics({"auc_train": auc_tr, "auc_valid": auc_va,
                        "gini_train": 2*auc_tr - 1, "gini_valid": 2*auc_va - 1})

print(f"AUC  train {auc_tr:.4f} | valid {auc_va:.4f}")
print(f"Gini train {2*auc_tr-1:.4f} | valid {2*auc_va-1:.4f} | árboles usados: {m.best_iteration_}")

In [0]:
%pip install xgboost -q

In [0]:
import xgboost as xgb, mlflow
from sklearn.metrics import roc_auc_score

params_x = dict(objective="binary:logistic", eval_metric="auc", tree_method="hist",
                enable_categorical=True, learning_rate=0.05, n_estimators=2000,
                max_depth=5, min_child_weight=10, subsample=0.8, colsample_bytree=0.5,
                reg_lambda=5.0, early_stopping_rounds=100, random_state=42, verbosity=0)

with mlflow.start_run(run_name="xgb_challenger"):
    mx = xgb.XGBClassifier(**params_x)
    mx.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=100)

    auc_tr_x = roc_auc_score(y_tr, mx.predict_proba(X_tr)[:, 1])
    auc_va_x = roc_auc_score(y_va, mx.predict_proba(X_va)[:, 1])
    mlflow.log_params({**params_x, "best_iteration": mx.best_iteration})
    mlflow.log_metrics({"auc_train": auc_tr_x, "auc_valid": auc_va_x,
                        "gini_train": 2*auc_tr_x - 1, "gini_valid": 2*auc_va_x - 1})

print(f"XGB  AUC train {auc_tr_x:.4f} | valid {auc_va_x:.4f} | Gini valid {2*auc_va_x-1:.4f} | árboles: {mx.best_iteration}")

Scorecard WoE + regresión logística

Es el modelo interpretable. Cada variable se divide en tramos (bins), cada tramo recibe un peso de riesgo (WoE) y la regresión combina esos pesos. Se agrega al mismo notebook, después de las celdas anteriores.

In [0]:
# Funciones. Los tramos y los pesos se aprenden solo con train. Los nulos forman su propio tramo, porque la falta de datos puede ser informativa. Aquí WoE = ln(%malos / %buenos), de modo que un valor más alto significa más riesgo y los coeficientes de la regresión deberían salir positivos:

def fit_bins(s, n_bins=10):
    """Cortes por cuantiles (numérica) o None (categórica / pocas categorías). Solo con train."""
    if s.dtype.name == "category" or s.nunique() <= 10:
        return None
    return np.unique(s.quantile(np.linspace(0, 1, n_bins + 1)).values[1:-1])

def asignar_bin(s, cortes):
    if cortes is None:
        return s.astype("object").fillna("NA").astype(str)
    b = pd.Series(np.digitize(s.values.astype(float), cortes).astype(str), index=s.index)
    b[s.isna()] = "NA"
    return b

def tabla_woe(b, y):
    d = pd.DataFrame({"bin": b.values, "y": y.values})
    g = d.groupby("bin")["y"].agg(malos="sum", n="count")
    g["buenos"] = g["n"] - g["malos"]
    k = len(g)
    pb = (g["buenos"] + 0.5) / (g["buenos"].sum() + 0.5 * k)   # +0.5 evita log(0)
    pm = (g["malos"] + 0.5) / (g["malos"].sum() + 0.5 * k)
    g["woe"] = np.log(pm / pb)
    g["iv"] = (pm - pb) * g["woe"]
    return g

def a_woe(X, cols):
    out = pd.DataFrame(index=X.index)
    for c in cols:
        out[c] = asignar_bin(X[c], bins[c]).map(tablas[c]).fillna(0.0)  # tramo no visto -> neutro
    return out

In [0]:
# WoE e IV (poder predictivo) de cada variable:
bins, tablas, iv = {}, {}, {}
for c in features:
    bins[c] = fit_bins(X_tr[c])
    t = tabla_woe(asignar_bin(X_tr[c], bins[c]), y_tr)
    tablas[c], iv[c] = t["woe"], t["iv"].sum()

iv_s = pd.Series(iv).sort_values(ascending=False)
display(iv_s.head(15).to_frame("IV").reset_index().rename(columns={"index": "variable"}))

In [0]:
# Seleccionar variables, entrenar y registrar:
from sklearn.linear_model import LogisticRegression

cand = iv_s[(iv_s >= 0.02) & (iv_s <= 0.5)].index.tolist()   # ordenadas por IV desc
W_c = a_woe(X_tr, cand)
sel = []
for c in cand:   # si una variable se parece demasiado a otra ya elegida, se descarta
    if all(abs(W_c[c].corr(W_c[k])) < 0.7 for k in sel):
        sel.append(c)
print(f"{len(cand)} candidatas -> {len(sel)} tras quitar correlación > 0.7")

W_tr, W_va = a_woe(X_tr, sel), a_woe(X_va, sel)
with mlflow.start_run(run_name="scorecard_woe_lr"):
    lr = LogisticRegression(C=0.1, max_iter=1000).fit(W_tr, y_tr)
    auc_tr_s = roc_auc_score(y_tr, lr.predict_proba(W_tr)[:, 1])
    auc_va_s = roc_auc_score(y_va, lr.predict_proba(W_va)[:, 1])
    mlflow.log_params({"n_variables": len(sel), "C": 0.1})
    mlflow.log_metrics({"auc_train": auc_tr_s, "auc_valid": auc_va_s,
                        "gini_train": 2*auc_tr_s - 1, "gini_valid": 2*auc_va_s - 1})

print(f"Scorecard AUC train {auc_tr_s:.4f} | valid {auc_va_s:.4f} | Gini valid {2*auc_va_s-1:.4f}")
print("coeficientes negativos (deberían ser 0):", int((lr.coef_ < 0).sum()))

In [0]:
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

def ajustar_scorecard(cand, nombre, umbral_corr=0.7, C=0.1):
    W_c = a_woe(X_tr, cand)
    sel = []
    for c in cand:                      # 1) quitar variables casi duplicadas
        if all(abs(W_c[c].corr(W_c[k])) < umbral_corr for k in sel):
            sel.append(c)
    n_corr = len(sel)
    while True:                         # 2) quitar variables con signo contrario al riesgo
        lr = LogisticRegression(C=C, max_iter=1000).fit(W_c[sel], y_tr)
        peor = min(zip(lr.coef_[0], sel))
        if peor[0] >= 0:
            break
        sel.remove(peor[1])
    W_va = a_woe(X_va, sel)
    auc_tr = roc_auc_score(y_tr, lr.predict_proba(W_c[sel])[:, 1])
    auc_va = roc_auc_score(y_va, lr.predict_proba(W_va)[:, 1])
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params({"n_variables": len(sel), "C": C, "umbral_corr": umbral_corr})
        mlflow.log_metrics({"auc_train": auc_tr, "auc_valid": auc_va,
                            "gini_train": 2*auc_tr - 1, "gini_valid": 2*auc_va - 1})
    print(f"{nombre}: {len(cand)} candidatas -> {n_corr} sin correlación -> {len(sel)} finales | "
          f"AUC train {auc_tr:.4f} valid {auc_va:.4f} | Gini valid {2*auc_va-1:.4f}")
    return sel, lr

sel_A, lr_A = ajustar_scorecard(iv_s[(iv_s >= 0.02) & (iv_s <= 0.5)].index.tolist(), "scorecard_A_sin_MEAN")
sel_B, lr_B = ajustar_scorecard(iv_s[iv_s >= 0.02].index.tolist(), "scorecard_B_con_MEAN")